# House Price Prediction

### PROBLEM STATEMENT

Accurately predicting house prices is a critical task for buyers, sellers, and real
estate professionals. This project uses the Ames Housing dataset to build a
regression model that predicts a house's sale price based on its physical
characteristics, quality ratings, and location.

The goal is to compare multiple regression models, engineer meaningful features
from the raw data, and tune the best-performing model to produce accurate,
interpretable price predictions.

**NOTE:** Update the file path in the data-loading cell below to point to your
copy of the Ames Housing / house prices training CSV (e.g. from the Kaggle
"House Prices - Advanced Regression Techniques" competition).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error

import joblib

ModuleNotFoundError: No module named 'pandas'

## 1. Load the Data

In [ ]:
home = pd.read_csv('house_prices.csv')  # <-- update path
home.head()

In [ ]:
home.info()

In [ ]:
home.describe()

## 2. Data Cleaning

Drop columns with a very high proportion of missing values, then handle
remaining missing values in the columns we keep.

In [ ]:
missing_pct = home.isnull().mean().sort_values(ascending=False)
print(missing_pct[missing_pct > 0])

In [ ]:
# Drop columns that are more than 40% missing
cols_to_drop = missing_pct[missing_pct > 0.4].index.tolist()
home_clean = home.drop(columns=cols_to_drop)

# Fill remaining numeric missing values with median, categorical with mode
for col in home_clean.columns:
    if home_clean[col].isnull().sum() > 0:
        if home_clean[col].dtype in ['int64', 'float64']:
            home_clean[col] = home_clean[col].fillna(home_clean[col].median())
        else:
            home_clean[col] = home_clean[col].fillna(home_clean[col].mode()[0])

print(home_clean.isnull().sum().sum(), "missing values remaining")

## 3. Outlier Removal (IQR method)

Applied to the key numerical columns to reduce the influence of extreme values
before feature engineering.

In [ ]:
def remove_outliers(df, columns):
    df = df.copy()
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        before = df.shape[0]
        df = df[(df[col] >= lower) & (df[col] <= upper)]
        after = df.shape[0]
        print(f'{col}: removed {before - after} outliers')
    return df

outlier_cols = ['GrLivArea', 'TotalBsmtSF', 'LotArea', '1stFlrSF', 'GarageArea']
outlier_cols = [c for c in outlier_cols if c in home_clean.columns]
home_clean = remove_outliers(home_clean, outlier_cols)
home_clean = home_clean.reset_index(drop=True)

## 4. Feature Engineering

New features that capture combined/derived information the raw columns don't
express on their own.

In [ ]:
home_clean['house_age'] = home_clean['YrSold'] - home_clean['YearBuilt']
home_clean['total_sqft'] = home_clean['TotalBsmtSF'] + home_clean['1stFlrSF'] + home_clean['2ndFlrSF']
home_clean['area_per_room'] = home_clean['GrLivArea'] / (home_clean['TotRmsAbvGrd'] + 1)
home_clean['garage_area_ratio'] = home_clean['GarageArea'] / (home_clean['GrLivArea'] + 1)
home_clean['overall_space'] = home_clean['GrLivArea'] + home_clean['GarageArea']

## 5. Target Transformation

`SalePrice` is right-skewed, so we log-transform it for training. Predictions
are reversed with `np.expm1()` before evaluation so metrics stay in real dollars.

In [ ]:
home_clean['Sale_Pricelog'] = np.log1p(home_clean['SalePrice'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(home_clean['SalePrice'], bins=50, color='teal', edgecolor='black')
axes[0].set_title('SalePrice - Before Log Transform')
axes[1].hist(home_clean['Sale_Pricelog'], bins=50, color='orange', edgecolor='black')
axes[1].set_title('SalePrice - After Log Transform')
plt.tight_layout()
plt.show()

## 6. Exploratory Data Analysis

In [ ]:
numeric_cols = home_clean.select_dtypes(include=['int64', 'float64']).columns.tolist()

plt.figure(figsize=(14, 10))
corr = home_clean[numeric_cols].corr()
top_corr = corr['SalePrice'].abs().sort_values(ascending=False).head(15).index
sns.heatmap(home_clean[top_corr].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap - Top Features vs SalePrice')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x='GrLivArea', y='SalePrice', data=home_clean, alpha=0.6, color='teal')
plt.title('GrLivArea vs SalePrice')
plt.show()

## 7. Encoding & Scaling

One-hot encode categorical columns, then scale all features with MinMaxScaler.

In [ ]:
cat_cols = home_clean.select_dtypes(include='object').columns.tolist()
train_encoded = pd.get_dummies(home_clean, columns=cat_cols, drop_first=True)

x = train_encoded.drop(columns=['SalePrice', 'Sale_Pricelog']).reset_index(drop=True)
y = train_encoded['Sale_Pricelog'].reset_index(drop=True)

scaler = MinMaxScaler()
x_scaled = scaler.fit_transform(x)
x_scaled = pd.DataFrame(x_scaled, columns=x.columns)

## 8. Recursive Feature Elimination (RFE)

Select the top 15 features to keep the model focused and reduce noise from
the large number of one-hot encoded columns.

In [ ]:
rfe = RFE(LinearRegression(), n_features_to_select=15)
rfe.fit(x_scaled, y)

selected_features = x_scaled.columns[rfe.support_]
print("Selected features:", list(selected_features))

x_rfe = x_scaled[selected_features]

## 9. Train/Test Split and Baseline Model Comparison

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x_rfe, y, test_size=0.2, random_state=42)

models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.001),
    'Random Forest': RandomForestRegressor(random_state=23),
    'XGBoost': XGBRegressor(random_state=23)
}

y_test_actual = np.expm1(y_test)
results = []

for name, model in models.items():
    model.fit(x_train, y_train)
    log_preds = model.predict(x_test)
    actual_preds = np.expm1(log_preds)

    r2 = r2_score(y_test_actual, actual_preds)
    mse = mean_squared_error(y_test_actual, actual_preds)
    rmse = np.sqrt(mse)

    results.append({'Model': name, 'R2': round(r2, 4), 'MSE': round(mse, 2), 'RMSE': round(rmse, 2)})

results_df = pd.DataFrame(results).sort_values('R2', ascending=False)
display(results_df)

## 10. Hyperparameter Tuning

Only the top performers get tuned. Ridge (or the best linear model) uses
GridSearchCV since its search space is small; the best tree-based model uses
RandomizedSearchCV since its search space is much larger.

In [ ]:
# --- Ridge Tuning (GridSearchCV) ---
ridge_params = {'alpha': [0.01, 0.1, 1, 10, 50, 100, 200, 500, 1000]}
ridge_grid = GridSearchCV(Ridge(), ridge_params, cv=5, scoring='r2', n_jobs=-1)
ridge_grid.fit(x_train, y_train)

print("Best Ridge alpha:", ridge_grid.best_params_)
print("Best Ridge CV R2:", round(ridge_grid.best_score_, 4))

In [ ]:
# --- XGBoost Tuning (RandomizedSearchCV) ---
xgb_params = {
    'n_estimators': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6, 7],
    'subsample': [0.6, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 1.0]
}
xgb_random = RandomizedSearchCV(
    XGBRegressor(random_state=23), xgb_params, n_iter=50, cv=5,
    scoring='r2', random_state=23, n_jobs=-1
)
xgb_random.fit(x_train, y_train)

print("Best XGBoost params:", xgb_random.best_params_)
print("Best XGBoost CV R2:", round(xgb_random.best_score_, 4))

## 11. Evaluate Tuned Models on the Test Set

In [ ]:
tuned_models = {
    'Ridge (tuned)': ridge_grid.best_estimator_,
    'XGBoost (tuned)': xgb_random.best_estimator_
}

for name, model in tuned_models.items():
    log_preds = model.predict(x_test)
    actual_preds = np.expm1(log_preds)
    r2 = r2_score(y_test_actual, actual_preds)
    rmse = np.sqrt(mean_squared_error(y_test_actual, actual_preds))
    print(f"{name}: R2={r2:.4f}, RMSE=${rmse:,.2f}")

## 12. Diagnostics — Actual vs Predicted, Residuals, Feature Importance

Using the tuned Ridge model as the final model (update this if XGBoost ends
up winning on your re-run).

In [ ]:
final_model = ridge_grid.best_estimator_

log_preds = final_model.predict(x_test)
actual_preds = np.expm1(log_preds)
actual_y = np.expm1(y_test)

plt.figure(figsize=(8, 6))
plt.scatter(actual_y, actual_preds, alpha=0.6, color='teal')
plt.plot([actual_y.min(), actual_y.max()], [actual_y.min(), actual_y.max()], 'r--', linewidth=2)
plt.xlabel('Actual Prices')
plt.ylabel('Predicted Prices')
plt.title('Actual vs Predicted House Prices - Tuned Ridge')
plt.tight_layout()
plt.show()

In [ ]:
residuals = actual_y - actual_preds

plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=30, color='teal', edgecolor='black', alpha=0.7)
plt.axvline(x=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Residuals (Actual - Predicted)')
plt.ylabel('Frequency')
plt.title('Residual Distribution - Tuned Ridge')
plt.tight_layout()
plt.show()

print(f"Mean residual: ${residuals.mean():,.2f}")
print(f"Std of residuals: ${residuals.std():,.2f}")

In [ ]:
coefficients = pd.Series(final_model.coef_, index=x_train.columns)
top_features = coefficients.sort_values(key=abs, ascending=False).head(10)

plt.figure(figsize=(10, 6))
top_features.sort_values().plot(kind='barh', color='teal', edgecolor='black', alpha=0.7)
plt.axvline(x=0, color='red', linestyle='--', linewidth=1)
plt.xlabel('Coefficient')
plt.ylabel('Feature')
plt.title('Top 10 Feature Coefficients (Tuned Ridge)')
plt.tight_layout()
plt.show()

## 13. Save the Model

In [ ]:
joblib.dump(final_model, 'ridge_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(list(x_rfe.columns), 'feature_names.pkl')            # RFE-selected features the model uses
joblib.dump(x_scaled.columns.tolist(), 'all_feature_names.pkl')  # full pre-RFE feature list the scaler expects
joblib.dump(x_scaled.median(), 'feature_medians.pkl')            # medians to fill features not exposed to the user

print("Model, scaler, feature names and medians saved successfully.")

## 14. Conclusion

This project built an end-to-end machine learning pipeline to predict house
prices. After data cleaning, feature engineering, and exploratory analysis,
five models were compared, and the top performer(s) were tuned with
GridSearchCV / RandomizedSearchCV.

Key findings:
- Overall quality and condition are the strongest drivers of price.
- House age negatively affects price — older houses are worth less.
- Engineered features (`total_sqft`, `house_age`) ranked among the most
  important predictors, confirming the feature engineering added real value.
- Log-transforming the target improved model behavior and produced residuals
  centered near zero with no major systematic bias.

The final tuned model and its supporting artifacts (scaler, feature list,
medians) were saved for use in a deployed prediction API.